### Desarrollo de modelos - Predicción de demanda eléctrica a 24 horas

Se implementará y evaluará los procedimientos definidos en el protocolo experimental.

El periodo de desarrollo (2015-2018) se utlizará para el entrenamiento del modelo y la validación temporal. Los datos del 2019 se reservarán para la evaluación final.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Identificar raíz del repositorio
project_root = Path.cwd()

if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent

# Cargar base analítica
data_path = (
    project_root
    / "data"
    / "processed"
    / "demanda_base_analitica.csv"
)

model_df = pd.read_csv(
    data_path,
    parse_dates=["utc_timestamp", "target_timestamp"]
)

# Predictores y variable objetivo
candidate_predictors = [
    "country",
    "demand_mw",
    "lag_48h",
    "lag_72h",
    "lag_168h",
    "rolling_mean_24h",
    "rolling_mean_168h",
    "target_hour",
    "target_day",
    "target_month"
]

target = "target_demand_mw"

# Comprobación inicial
assert all(
    col in model_df.columns
    for col in candidate_predictors + [target]
)

print("Base cargada:", model_df.shape)
print("Predictores:", len(candidate_predictors))
print("Periodos:", model_df["period"].value_counts(dropna=False).to_dict())

Base cargada: (131472, 25)
Predictores: 10
Periodos: {'development': 105120, 'holdout': 26280, nan: 72}


In [4]:
validation_windows = [
    ("W1", "2016-01-01", "2017-01-01"),
    ("W2", "2017-01-01", "2018-01-01"),
    ("W3", "2018-01-01", "2019-01-01")
]

validation_partitions = {}

for name, start_date, end_date in validation_windows:

    start = pd.Timestamp(start_date, tz="UTC")
    end = pd.Timestamp(end_date, tz="UTC")

    train_mask = (
        (model_df["period"] == "development")
        & (model_df["utc_timestamp"] < start)
        & (model_df["target_timestamp"] < start)
    )

    validation_mask = (
        (model_df["period"] == "development")
        & (model_df["utc_timestamp"] >= start)
        & (model_df["utc_timestamp"] < end)
    )

    # Mismo criterio de casos completos para todos los modelos
    required = candidate_predictors + [target]

    train_df = (
        model_df.loc[train_mask]
        .dropna(subset=required)
        .copy()
    )

    validation_df = (
        model_df.loc[validation_mask]
        .dropna(subset=required)
        .copy()
    )

    assert not train_df.empty
    assert not validation_df.empty

    assert (
        train_df["target_timestamp"].max()
        < validation_df["utc_timestamp"].min()
    )

    validation_partitions[name] = {
        "train": train_df,
        "validation": validation_df
    }

    print(
        f"{name}: "
        f"Train={len(train_df):,} | "
        f"Validation={len(validation_df):,}"
    )

W1: Train=25,484 | Validation=26,352
W2: Train=51,836 | Validation=26,087
W3: Train=77,923 | Validation=25,685


### Modelos de referencia

Se evaluarán:

- Persistencia: Predice que la demanda de dentro de 24 horas será igual a la demanda actual (`demand_mw`).
- Estacional semanal: Predice que la demanda será igual a la observada una semana antes (`lag_168h`).

La siguiente función calcula MAE, RMSE y NMAE por país, también el NMAE macro. 

In [5]:
def evaluate_predictions(train_df, validation_df, predictions, model_name, window_name):
    results = []

    for country in ["DE", "ES", "FR"]:
        train_country = train_df[train_df["country"] == country]
        valid_country = validation_df[validation_df["country"] == country]

        y_true = valid_country[target].to_numpy()
        y_pred = predictions.loc[valid_country.index].to_numpy()

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        training_mean = train_country[target].mean()
        nmae = mae / training_mean

        results.append({
            "window": window_name,
            "model": model_name,
            "country": country,
            "n": len(valid_country),
            "MAE": mae,
            "RMSE": rmse,
            "NMAE": nmae
        })

    results_df = pd.DataFrame(results)

    macro_row = {
        "window": window_name,
        "model": model_name,
        "country": "Macro",
        "n": results_df["n"].sum(),
        "MAE": np.nan,
        "RMSE": np.nan,
        "NMAE": results_df["NMAE"].mean()
    }

    return pd.concat(
        [results_df, pd.DataFrame([macro_row])],
        ignore_index=True
    )

### Evaluación de los baselines

In [6]:
baseline_results = []

for window_name, partition in validation_partitions.items():
    train_df = partition["train"]
    validation_df = partition["validation"]

    # Baseline 1: persistencia
    pred_persistence = validation_df["demand_mw"]

    results_persistence = evaluate_predictions(
        train_df,
        validation_df,
        pred_persistence,
        "Persistence",
        window_name
    )

    # Baseline 2: estacional semanal
    pred_weekly = validation_df["lag_168h"]

    results_weekly = evaluate_predictions(
        train_df,
        validation_df,
        pred_weekly,
        "Weekly Seasonal",
        window_name
    )

    baseline_results.extend([
        results_persistence,
        results_weekly
    ])

baseline_results_df = pd.concat(
    baseline_results,
    ignore_index=True
)

display(
    baseline_results_df.round(4)
)

,window,model,country,n,MAE,RMSE,NMAE
0,W1,Persistence,DE,8784,4492.1990,7018.4174,0.0820
1,W1,Persistence,ES,8784,1789.8759,2760.4697,0.0630
2,W1,Persistence,FR,8784,3091.1412,4592.4669,0.0580
3,W1,Persistence,Macro,26352,NaN,NaN,0.0677
4,W1,Weekly Seasonal,DE,8784,2207.8969,4006.1230,0.0403
5,W1,Weekly Seasonal,ES,8784,1285.2696,2013.7848,0.0453
6,W1,Weekly Seasonal,FR,8784,3458.4820,4974.0852,0.0648
7,W1,Weekly Seasonal,Macro,26352,NaN,NaN,0.0501
8,W2,Persistence,DE,8760,4550.1024,7006.8546,0.0826
9,W2,Persistence,ES,8760,1841.5156,2781.0271,0.0647


Combinaciones de ventana y modelo, con métricas para DE, ES, FR y macro. El valor de NMAE macro es nuestra métrica principal de comparación. Los baselines se evaluan sobre las mismas observaciones completas que usaremos para los modelos predictivos, evitando así, comparaciones injustas por diferencias en el total de casos.

## M1 - Regresión Regularizada Ridge

La preparación de variables será:

- A las categóricas como: `country`,  `target_hour`, `target_day`, `target_month`, se les tratará como variables dummy.
- A las numéricas como: `demand_mw`, `lag_48h`, `lag_72h`, `lag_168h`, `rolling_mean_24h`, `rolling_mean_168h`, se les tratará con estandarización.

In [7]:
categorical_features = [
    "country",
    "target_hour",
    "target_day",
    "target_month"
]

numeric_features = [
    "demand_mw",
    "lag_48h",
    "lag_72h",
    "lag_168h",
    "rolling_mean_24h",
    "rolling_mean_168h"
]

ridge_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", ridge_preprocessor),
        ("model", Ridge())
    ]
)

### Ajuste del hipeparámetro `alpha`

Tal parámetro controla la intensidad de la regulación. Un valor bajo se aproxima a una regresión lineal, uno mayor penaliza más los coeficientes.

Se probarán 15 valores en escala logarítmica. Cada configuración se evaluará en W!, W2 y W3

In [8]:
alpha_grid = np.logspace(-3, 4, 15)

ridge_tuning_results = []

for alpha in alpha_grid:
    for window_name, partition in validation_partitions.items():
        train_df = partition["train"]
        validation_df = partition["validation"]

        X_train = train_df[candidate_predictors]
        y_train = train_df[target]

        X_valid = validation_df[candidate_predictors]

        ridge_pipeline.set_params(model__alpha=alpha)

        ridge_pipeline.fit(X_train, y_train)

        predictions = pd.Series(
            ridge_pipeline.predict(X_valid),
            index=validation_df.index
        )

        metrics = evaluate_predictions(
            train_df,
            validation_df,
            predictions,
            f"Ridge alpha={alpha:.5g}",
            window_name
        )

        macro_nmae = metrics.loc[
            metrics["country"] == "Macro", "NMAE"
        ].iloc[0]

        ridge_tuning_results.append({
            "alpha": alpha,
            "window": window_name,
            "NMAE_macro": macro_nmae
        })

ridge_tuning_df = pd.DataFrame(ridge_tuning_results)

ridge_alpha_summary = (
    ridge_tuning_df
    .groupby("alpha", as_index=False)
    .agg(
        mean_NMAE=("NMAE_macro", "mean"),
        std_NMAE=("NMAE_macro", "std")
    )
    .sort_values("mean_NMAE")
    .reset_index(drop=True)
)

display(ridge_alpha_summary.round(5))

,alpha,mean_NMAE,std_NMAE
0,316.22777,0.04282,0.00117
1,100.00000,0.04283,0.00124
2,31.62278,0.04291,0.00120
3,10.00000,0.04296,0.00116
4,3.16228,0.04297,0.00115
5,1.00000,0.04298,0.00115
6,0.31623,0.04298,0.00115
7,0.10000,0.04298,0.00114
8,0.03162,0.04298,0.00114
9,0.01000,0.04298,0.00114


### Selección de `alpha`

Se selecciona el valor con menor `NMAE macro` promedio en las tres ventanas.

In [9]:
best_alpha = ridge_alpha_summary.loc[0, "alpha"]

print(f"Mejor alpha: {best_alpha:.5g}")
print(
    "NMAE macro promedio:",
    round(ridge_alpha_summary.loc[0, "mean_NMAE"], 5)
)

Mejor alpha: 316.23
NMAE macro promedio: 0.04282


### Evaluación del Ridge seleccionado

Volvemos a ajustar el Ridge con el `alpha` elegido en cada ventana y guardamos sus métricas por país.

In [10]:
ridge_results = []

for window_name, partition in validation_partitions.items():
    train_df = partition["train"]
    validation_df = partition["validation"]

    X_train = train_df[candidate_predictors]
    y_train = train_df[target]

    X_valid = validation_df[candidate_predictors]

    ridge_pipeline.set_params(model__alpha=best_alpha)
    ridge_pipeline.fit(X_train, y_train)

    predictions = pd.Series(
        ridge_pipeline.predict(X_valid),
        index=validation_df.index
    )

    metrics = evaluate_predictions(
        train_df,
        validation_df,
        predictions,
        "Ridge",
        window_name
    )

    ridge_results.append(metrics)

ridge_results_df = pd.concat(
    ridge_results,
    ignore_index=True
)

display(ridge_results_df.round(4))

,window,model,country,n,MAE,RMSE,NMAE
0,W1,Ridge,DE,8784,2240.1721,3318.0079,0.0409
1,W1,Ridge,ES,8784,1156.7657,1626.6156,0.0407
2,W1,Ridge,FR,8784,2316.7857,3203.8817,0.0434
3,W1,Ridge,Macro,26352,NaN,NaN,0.0417
4,W2,Ridge,DE,8760,2325.2877,3476.8884,0.0422
5,W2,Ridge,ES,8760,1255.6361,1751.8319,0.0441
6,W2,Ridge,FR,8567,2259.1954,3099.8137,0.0419
7,W2,Ridge,Macro,26087,NaN,NaN,0.0428
8,W3,Ridge,DE,8736,2373.9591,3458.3714,0.0428
9,W3,Ridge,ES,8736,1233.4642,1702.4829,0.0431


### Comparación inicial

Reunimos las métricas de los tres procedimientos para comparar sus resultados.

In [11]:
all_results_df = pd.concat(
    [
        baseline_results_df,
        ridge_results_df
    ],
    ignore_index=True
)

macro_comparison = (
    all_results_df[all_results_df["country"] == "Macro"]
    .pivot(
        index="window",
        columns="model",
        values="NMAE"
    )
)

display(macro_comparison.round(5))

print("\nNMAE macro promedio por modelo:")
display(
    macro_comparison.mean()
    .sort_values()
    .to_frame("Mean NMAE")
    .round(5)
)

model,Persistence,Ridge,Weekly Seasonal
window,,,
W1,0.06766,0.04169,0.05014
W2,0.06779,0.04275,0.05415
W3,0.06829,0.04402,0.05693



NMAE macro promedio por modelo:


,Mean NMAE
model,
Ridge,0.04282
Weekly Seasonal,0.05374
Persistence,0.06792


Ridge supera ambos baselines en las tres ventanas temporales y no solamente en el promedio global.

Se debe tener en cuenta la diferencia entre paises y la variación del desempeño entre ventanas. Además, debe considerarse en la interpretación de W2 y W3, que Francia tiene menos observaciones que Alemania y España debido a los valores faltantes. Pero por ahora, las comparaciones son válidas, aunque se debe tener presente esta diferencia de tamaño muestral.

### Interpretación de los resultados

#### Comparación general

Ridge obtiene un `NMAE macro` promedio de $0.04282$, frente a $0.05374$ del baseline estacional semanal y $0.06792$ de prevalencia.

En cuanto a persistencia, Ridge reduce el `NMAE macro` cerca de un $37\%$. Mientras que en el baseline semanal, lo reduce cerca de un $20.3\%$, lo que implica que Ridge consigue mejorar una referencia que ya contiene un patrón estacional de una semana.

Si se observa la tabla de NMAE por ventanas, se aprecia que la ventaja de Ridge se mantiene en las tres ventanas. Sin embargo, el NMAE aumenta crecientemente de W1 a W3, lo que puede decir que la predicción se vuelve más dificil en periodos posteriores, aunque Ridge conserva ventaja clara sobre las referencias.

Con respecto al baseline semanal, la reducción también se mantiene, cerca de un $16.9\%$ en W1, $21\%$ en W2 y $22.6\%$ en W3.

#### Comparación entre países

Los baselines presentan patrones distintos.

- En Alemania, la persistencia tiene un `NMAE` relativamente alto, de un $8.2-8.3\%$, pero el baseline semanal mejora a un $4.0-4.6\%$.
- En España, el baseline semanal también mejora la persistencia, con un `NMAE` entre $4.5-5.1\%$.
- En Francia, por el contrario, el baseline semanal de $6.48-7.76\%$ es peor que en la persistencia, un $5.6-5.8\%$.

Esto implica que la referencia semanal funciona distinto por país. Pero aún no se puede concluir cómo se comporta Ridge en cada uno, ya que solo se tienen los promedios macro.

In [12]:
country_comparison = (
    all_results_df
    .groupby(["country", "model"])["NMAE"]
    .mean()
    .unstack("model")
)

display(country_comparison.round(5))

model,Persistence,Ridge,Weekly Seasonal
country,,,
DE,0.08259,0.04196,0.04357
ES,0.06385,0.04267,0.04787
FR,0.05731,0.04383,0.06978
Macro,0.06792,0.04282,0.05374


`NMAE` promedio de cada modelo por país.

In [13]:
display(
    ridge_results_df
    .sort_values(["window", "country"])
    .round(4)
)

,window,model,country,n,MAE,RMSE,NMAE
0,W1,Ridge,DE,8784,2240.1721,3318.0079,0.0409
1,W1,Ridge,ES,8784,1156.7657,1626.6156,0.0407
2,W1,Ridge,FR,8784,2316.7857,3203.8817,0.0434
3,W1,Ridge,Macro,26352,NaN,NaN,0.0417
4,W2,Ridge,DE,8760,2325.2877,3476.8884,0.0422
5,W2,Ridge,ES,8760,1255.6361,1751.8319,0.0441
6,W2,Ridge,FR,8567,2259.1954,3099.8137,0.0419
7,W2,Ridge,Macro,26087,NaN,NaN,0.0428
8,W3,Ridge,DE,8736,2373.9591,3458.3714,0.0428
9,W3,Ridge,ES,8736,1233.4642,1702.4829,0.0431


Métricas completas de Ridge por ventana y país.

Finalmente, con respecto a la elección del `alpha`, el mejor valor encontrado fué $\alpha=316.22777$. Sin embargo, no existe una diferencia importante entre muchos de los valores intermedios desde $\alpha=0.1$ hasta $\alpha=316$, en donde el `NMAE macro` se mantiene estable.

Esto sugiere que el rendimiento de Ridge no depende en su mayoría de una elección muy precisa del `alpha` dentro de ese rango. Lo que es una buena señal de estabilidad del hiperparámetro. No obstante, se mantiene la elección de $\alpha=316.22777$.

## M2 - Árbol de regresión podado

Se evaluará si un modelo que captura relaciones no lineales e interacciones entre variables es capaz de mejorar el desempeño de Ridge.

Se usará `DesisionTreeRegressor` con poda, por costo de complejidad (`ccp_alpha`). Se mantienen las mismas variables, ventanas temporales y observaciones completas para una justa comparación.

Se usarán los mismos predictores que en Ridge. Las variables categóricas como variables dummy, pero las seis variables numéricas de demanda no serán estandarizadas, ya que no es necesario en un árbol de decisión.

### Configuración del árbol

In [14]:
from sklearn.tree import DecisionTreeRegressor

def make_tree_pipeline(ccp_alpha):
    tree_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features
            )
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            (
                "model",
                DecisionTreeRegressor(
                    random_state=42,
                    ccp_alpha=ccp_alpha
                )
            )
        ]
    )

### Construcción de configuraciones de poda

El parámetro `cpp_alpha` indica cuánto se poda el árbol. Un valor de $0$ indica un árbol sin poda por costo de complejidad, mientras que valores mayores, indican una mayor penalización por complejidad (y generalmente árboles más pequeños).

Para seleccionar los valores candidatos, se usarán rutas de poda calculadas con los datos de entrenamiento de cada ventana.



### Obtener valores candidatos

In [16]:
pruning_alphas = []

for window_name, partition in validation_partitions.items():
    train_df = partition["train"]

    X_train = train_df[candidate_predictors]
    y_train = train_df[target]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric_features),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features
            )
        ]
    )

    X_train_transformed = preprocessor.fit_transform(X_train)

    tree_probe = DecisionTreeRegressor(random_state=42)

    pruning_path = tree_probe.cost_complexity_pruning_path(
        X_train_transformed,
        y_train
    )

    positive_alphas = pruning_path.ccp_alphas[
        pruning_path.ccp_alphas > 0
    ]

    pruning_alphas.extend(positive_alphas.tolist())

# Reunir y depurar los valores candidatos
all_positive_alphas = np.asarray(pruning_alphas, dtype=float)

all_positive_alphas = all_positive_alphas[
    np.isfinite(all_positive_alphas)
]

if len(all_positive_alphas) == 0:
    raise ValueError("No se encontraron valores positivos de poda.")

tree_alpha_grid = np.unique(
    np.concatenate([
        [0.0],
        np.quantile(
            all_positive_alphas,
            np.linspace(0, 1, 19)
        )
    ])
)

print("Configuraciones de poda:", len(tree_alpha_grid))
print("Valores candidatos:")
print(tree_alpha_grid)

Configuraciones de poda: 20
Valores candidatos:
[0.00000000e+00 6.41659074e-06 3.10562992e-03 1.47838251e-02
 4.17549574e-02 9.39694270e-02 1.87627530e-01 3.43819932e-01
 6.11253674e-01 1.02890049e+00 1.70970060e+00 2.83359055e+00
 4.65972931e+00 7.72783389e+00 1.29018904e+01 2.23739783e+01
 4.14460514e+01 8.78027762e+01 2.56865784e+02 1.66150129e+08]


### Ajuste y evaluación del árbol

In [17]:
tree_tuning_results = []

for alpha in tree_alpha_grid:
    for window_name, partition in validation_partitions.items():
        train_df = partition["train"]
        validation_df = partition["validation"]

        X_train = train_df[candidate_predictors]
        y_train = train_df[target]

        X_valid = validation_df[candidate_predictors]

        tree_pipeline = make_tree_pipeline(alpha)

        tree_pipeline.fit(X_train, y_train)

        predictions = pd.Series(
            tree_pipeline.predict(X_valid),
            index=validation_df.index
        )

        metrics = evaluate_predictions(
            train_df,
            validation_df,
            predictions,
            f"Decision Tree alpha={alpha:.6g}",
            window_name
        )

        macro_nmae = metrics.loc[
            metrics["country"] == "Macro", "NMAE"
        ].iloc[0]

        tree_tuning_results.append({
            "ccp_alpha": alpha,
            "window": window_name,
            "NMAE_macro": macro_nmae
        })

tree_tuning_df = pd.DataFrame(tree_tuning_results)

tree_alpha_summary = (
    tree_tuning_df
    .groupby("ccp_alpha", as_index=False)
    .agg(
        mean_NMAE=("NMAE_macro", "mean"),
        std_NMAE=("NMAE_macro", "std")
    )
    .sort_values("mean_NMAE")
    .reset_index(drop=True)
)

display(tree_alpha_summary.round(6))

,ccp_alpha,mean_NMAE,std_NMAE
0,2.568658e+02,0.036068,0.001228
1,8.780278e+01,0.036907,0.001245
2,4.144605e+01,0.037399,0.001127
3,2.237398e+01,0.037819,0.000964
4,1.290189e+01,0.038070,0.000814
5,7.727834e+00,0.038242,0.000729
6,4.659729e+00,0.038356,0.000676
7,2.833591e+00,0.038428,0.000650
8,1.709701e+00,0.038483,0.000630
9,1.028900e+00,0.038517,0.000614


### Seleccionar nivel de poda

In [18]:
best_tree_alpha = tree_alpha_summary.loc[0, "ccp_alpha"]

print(f"Mejor ccp_alpha: {best_tree_alpha:.8g}")
print(
    "NMAE macro promedio:",
    round(tree_alpha_summary.loc[0, "mean_NMAE"], 5)
)

Mejor ccp_alpha: 256.86578
NMAE macro promedio: 0.03607


### Evaluación del árbol seleccionado

Se verá ahora los resultados y la complejidad del árbol seleccionado.

In [19]:
tree_results = []
tree_complexity = []

for window_name, partition in validation_partitions.items():
    train_df = partition["train"]
    validation_df = partition["validation"]

    X_train = train_df[candidate_predictors]
    y_train = train_df[target]

    X_valid = validation_df[candidate_predictors]

    tree_pipeline = make_tree_pipeline(best_tree_alpha)

    tree_pipeline.fit(X_train, y_train)

    predictions = pd.Series(
        tree_pipeline.predict(X_valid),
        index=validation_df.index
    )

    metrics = evaluate_predictions(
        train_df,
        validation_df,
        predictions,
        "Decision Tree",
        window_name
    )

    tree_results.append(metrics)

    fitted_tree = tree_pipeline.named_steps["model"]

    tree_complexity.append({
        "window": window_name,
        "depth": fitted_tree.get_depth(),
        "leaves": fitted_tree.get_n_leaves()
    })

tree_results_df = pd.concat(tree_results, ignore_index=True)
tree_complexity_df = pd.DataFrame(tree_complexity)

display(tree_results_df.round(4))
display(tree_complexity_df)

,window,model,country,n,MAE,RMSE,NMAE
0,W1,Decision Tree,DE,8784,1794.7498,3158.3359,0.0328
1,W1,Decision Tree,ES,8784,997.8611,1643.4047,0.0351
2,W1,Decision Tree,FR,8784,2325.5112,3472.6411,0.0436
3,W1,Decision Tree,Macro,26352,NaN,NaN,0.0372
4,W2,Decision Tree,DE,8760,1952.9760,3484.1465,0.0354
5,W2,Decision Tree,ES,8760,960.4637,1562.5509,0.0338
6,W2,Decision Tree,FR,8567,2137.5979,3224.5218,0.0397
7,W2,Decision Tree,Macro,26087,NaN,NaN,0.0363
8,W3,Decision Tree,DE,8736,1819.1375,3181.2323,0.0328
9,W3,Decision Tree,ES,8736,814.0384,1404.0375,0.0285


,window,depth,leaves
0,W1,23,2785
1,W2,28,3550
2,W3,30,3965


### Comparación con los modelos anteriores

In [20]:
all_results_df = pd.concat(
    [
        baseline_results_df,
        ridge_results_df,
        tree_results_df
    ],
    ignore_index=True
)

macro_comparison = (
    all_results_df[all_results_df["country"] == "Macro"]
    .pivot(
        index="window",
        columns="model",
        values="NMAE"
    )
)

display(macro_comparison.round(5))

print("\nNMAE macro promedio:")
display(
    macro_comparison.mean()
    .sort_values()
    .to_frame("Mean NMAE")
    .round(5)
)

model,Decision Tree,Persistence,Ridge,Weekly Seasonal
window,,,,
W1,0.03717,0.06766,0.04169,0.05014
W2,0.03629,0.06779,0.04275,0.05415
W3,0.03474,0.06829,0.04402,0.05693



NMAE macro promedio:


,Mean NMAE
model,
Decision Tree,0.03607
Ridge,0.04282
Weekly Seasonal,0.05374
Persistence,0.06792


### Interpretación de los resultados

El árbol de decisión con poda tuvo el promedio `NMAE macro` más bajo hasta el momento, de $0.03607$, superando al de Ridge de $0.04282$, al estacional semanal de $0.05374$ y al de persistencia de $0.06792$.

Se logró mejorar la precisión del modelo en un $15.8\%$ con respecto a Ridge y la mejora es consistente en las tres ventanas de validación.

Por otro lado, los valores del `NMAE macro` disminuyen de W1 a W3, lo que indica un rendimiento predictivo estable y mejorable a lo largo de los periodos evaluados.

El parámetro de poda elegido es el `ccp_alpha`=$256.86578$ que genera el `NMAE macro` más bajo. Los árboles obtenidos tienen profundidades de $23$, $28$ y $30$, y tienen $2785$, $3550$ y $3965$ hojas respectivamente. La complejidad de estos árboles aumenta mediante crece el tamaño del conjunto de entrenamiento.

Cuando se buscaba ajustar la poda, se obtuvo un valor de `ccp_alpha` bastante grande del orden de $10^8$, el cual produce un rendimiento notoriamente peor (`NMAE`=$0.339291$). Lo que dice que una poda excesiva puede producir un ajuste deficiente.

### Cuánto afecta la poda

Para ver cuanto afecta la poda en rendimiento predictivo, se cuantificará el efecto con un `ccp_alpha`=$0$ y se comparará con el ajuste con el `ccp_alpha`=$256.86578$ elegido.

In [21]:
unpruned_nmae = tree_alpha_summary.loc[
    tree_alpha_summary["ccp_alpha"] == 0,
    "mean_NMAE"
].iloc[0]

pruned_nmae = tree_alpha_summary.loc[0, "mean_NMAE"]

pruning_improvement = (
    (unpruned_nmae - pruned_nmae) / unpruned_nmae
) * 100

print(f"NMAE sin poda: {unpruned_nmae:.5f}")
print(f"NMAE con poda seleccionada: {pruned_nmae:.5f}")
print(f"Reducción relativa del NMAE: {pruning_improvement:.2f}%")

NMAE sin poda: 0.03857
NMAE con poda seleccionada: 0.03607
Reducción relativa del NMAE: 6.49%


Los valores obtenidos del `NMAE` sin poda de $0.03857$ y del `NMAE` con poda de $0.03607$, indican que aparte de reducir complejidad, también mejoran la capacidad predictiva en el conjunto de validación.

Por otro lado, los árboles tienen entre $2785$ y $3965$ hojas. Si bien sus profundidades son moderadas ($23$-$30$), la cantidad de hojas si es considerable. Esto puede deberse a que la complejidad aumenta junto con el tamaño muestral del entrenamiento y que incluso el rendimiento se mantiene o mejora levemente entre ventanas. Sin embargo, M2 sigue siendo un modelo complicado de interpretar. Finalmente, se puede decir que la configuración que se seleccionó es razonable predictivamente, pero es limitada en cuanto a interpretavilidad en la práctica.

Por ahora, M2 ofrece el mejor rendimiento predictivo entre todos los modelos evaluados.